In [1]:
!nvidia-smi

Tue Jan 28 07:45:49 2025       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              12W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [2]:
!pip install deepface opencv-python numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 8.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 60.7 MB/s eta 0:00:00
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114249 sha256=9387dc13dd26dd2a2e55fdc2b96634657e60174b239b930449d31cfbf6b074b5
  Stored in directory: /root/.cache/pip/wheels/46/54/24/1624fd5b8674eb1188623f7e8e17cdf7c0f6c24b609dfb8a89
Successfully built fire


In [3]:
from deepface import DeepFace
import cv2
import os
import time
from collections import Counter

25-01-28 07:46:53 - Directory /root/.deepface has been created
25-01-28 07:46:53 - Directory /root/.deepface/weights has been created


In [4]:
# Define custom emotion mapping
emotion_mapping = {
    'happy': 'Engaged',
    'surprise': 'Engaged',
    'neutral': 'Engaged',
    'sad': 'Distracted',
    'disgust': 'Distracted',
    'fear': 'Confused',
    'angry': 'Confused'
}

# Paths
video_path = "/content/drive/MyDrive/face_detection/class3.mp4"  # Input video
output_video_path = "/content/drive/MyDrive/face_detection/output_video7.mp4"  # Output video

# Open video file
cap = cv2.VideoCapture(video_path)

# Check if video opened successfully
if not cap.isOpened():
    print("Error: Could not open video.")
    exit()

# Get video properties
fps = cap.get(cv2.CAP_PROP_FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Define codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

# Start total time tracking
start_total_time = time.time()

# Store detected emotions
emotion_counts = Counter()

# Process every 3rd frame
frame_count = 0
process_every_n_frames = 3  # 🔹 Process every 3rd frame

while True:
    ret, frame = cap.read()
    if not ret:
        break  # Stop if video ends

    # Process only every 3rd frame
    if frame_count % process_every_n_frames == 0:
        start_time = time.time()

        # Analyze emotions
        result = DeepFace.analyze(frame, actions=['emotion'], enforce_detection=False)

        # Process detected faces
        for face in result:
            emotion = face['dominant_emotion']
            confidence = face['emotion'][emotion]

            # Map to custom emotion category
            custom_emotion = emotion_mapping.get(emotion, "Unknown")

            # Track emotion counts
            if custom_emotion != "Unknown":
                emotion_counts[custom_emotion] += 1

            # Get face coordinates
            region = face['region']
            x, y, w, h = region['x'], region['y'], region['w'], region['h']

            # Annotate frame with emotion label
            label = f"{custom_emotion} ({confidence*100:.2f}%)"
            cv2.putText(frame, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2, cv2.LINE_AA)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        # Time taken for this frame
        time_taken = time.time() - start_time
        print(f"Frame {frame_count + 1}: Processed (Time taken = {time_taken:.2f} sec)")

    # Write frame to output video (even if not processed)
    out.write(frame)

    frame_count += 1

# Release resources
cap.release()
out.release()
cv2.destroyAllWindows()

# Calculate total time
total_time_taken = time.time() - start_total_time

# Determine overall dominant emotion
dominant_emotion = emotion_counts.most_common(1)[0][0] if emotion_counts else "Unknown"

# Print summary
print(f"\n🎯 Total Processing Time: {total_time_taken:.2f} sec")
print(f"📌 Overall Dominant Emotion: {dominant_emotion}")
print(f"✅ Video processing complete. Output saved to '{output_video_path}'")


25-01-28 07:47:30 - facial_expression_model_weights.h5 will be downloaded...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: /root/.deepface/weights/facial_expression_model_weights.h5
100%|██████████| 5.98M/5.98M [00:00<00:00, 160MB/s]


Frame 1: Processed (Time taken = 22.77 sec)
Frame 4: Processed (Time taken = 18.10 sec)
Frame 7: Processed (Time taken = 19.91 sec)
Frame 10: Processed (Time taken = 19.27 sec)
Frame 13: Processed (Time taken = 18.01 sec)
Frame 16: Processed (Time taken = 15.46 sec)
Frame 19: Processed (Time taken = 20.31 sec)
Frame 22: Processed (Time taken = 18.24 sec)
Frame 25: Processed (Time taken = 18.04 sec)
Frame 28: Processed (Time taken = 19.78 sec)
Frame 31: Processed (Time taken = 17.78 sec)
Frame 34: Processed (Time taken = 17.92 sec)
Frame 37: Processed (Time taken = 19.76 sec)
Frame 40: Processed (Time taken = 18.35 sec)
Frame 43: Processed (Time taken = 18.19 sec)
Frame 46: Processed (Time taken = 21.60 sec)
Frame 49: Processed (Time taken = 18.36 sec)
Frame 52: Processed (Time taken = 18.26 sec)
Frame 55: Processed (Time taken = 20.32 sec)
Frame 58: Processed (Time taken = 18.39 sec)
Frame 61: Processed (Time taken = 18.60 sec)
Frame 64: Processed (Time taken = 20.48 sec)
Frame 67: Pro

In [6]:
from deepface import DeepFace
import cv2
import os
import time

# Define emotion mapping
emotion_mapping = {
    'happy': 'Engaged',
    'surprise': 'Engaged',
    'neutral': 'Distracted',
    'sad': 'Distracted',
    'disgust': 'Distracted',
    'fear': 'Confused',
    'angry': 'Confused'
}

# Input and output directories
input_path = "/content/drive/MyDrive/face_detection/Test_Images"  # Change this to a single image path or folder
output_path = "/content/drive/MyDrive/face_detection/Detected_Images"

# Create output directory if it doesn't exist
os.makedirs(output_path, exist_ok=True)

# Process a single image or multiple images
if os.path.isdir(input_path):
    image_files = [os.path.join(input_path, f) for f in os.listdir(input_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
else:
    image_files = [input_path]

# Process each image
for img_path in image_files:
    print(f"Processing: {img_path}")
    start_time = time.time()

    # Load image
    img = cv2.imread(img_path)

    # Analyze emotions in the image
    results = DeepFace.analyze(img, actions=['emotion'], enforce_detection=False)

    # Process detected faces
    for face in results:
        emotion = face['dominant_emotion']
        confidence = face['emotion'][emotion]

        # Map to custom emotion category
        custom_emotion = emotion_mapping.get(emotion, "Unknown")

        # Get face bounding box
        x, y, w, h = face['region']['x'], face['region']['y'], face['region']['w'], face['region']['h']

        # Draw bounding box and label
        label = f"{custom_emotion} ({confidence*100:.2f}%)"
        cv2.putText(img, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Save annotated image
    output_filename = os.path.join(output_path, os.path.basename(img_path))
    cv2.imwrite(output_filename, img)

    print(f"✅ Processed in {time.time() - start_time:.2f} sec. Saved to {output_filename}")

print("\n🎯 All images processed successfully!")

Processing: /content/drive/MyDrive/face_detection/Test_Images/img5.jpg
✅ Processed in 1.86 sec. Saved to /content/drive/MyDrive/face_detection/Detected_Images/img5.jpg
Processing: /content/drive/MyDrive/face_detection/Test_Images/img3.jpg
✅ Processed in 1.17 sec. Saved to /content/drive/MyDrive/face_detection/Detected_Images/img3.jpg
Processing: /content/drive/MyDrive/face_detection/Test_Images/img4.jpg
✅ Processed in 1.42 sec. Saved to /content/drive/MyDrive/face_detection/Detected_Images/img4.jpg
Processing: /content/drive/MyDrive/face_detection/Test_Images/img1.jpg
✅ Processed in 1.03 sec. Saved to /content/drive/MyDrive/face_detection/Detected_Images/img1.jpg
Processing: /content/drive/MyDrive/face_detection/Test_Images/img2.jpg
✅ Processed in 0.73 sec. Saved to /content/drive/MyDrive/face_detection/Detected_Images/img2.jpg

🎯 All images processed successfully!
